<a href="https://colab.research.google.com/github/esthy13/Water-Level-Distribution/blob/main/water_level_prediction_draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water level prediction using ERA5 (Master-level baseline)

This notebook:
- Downloads the dataset
- Normalizes ERA5 data
- Builds features (ephemerides + time + ERA5 + lat/lon)
- Trains a neural network (MLP)
- Evaluates RMSE on test set (2020)

**Constraint respected:** no past water level values are used as inputs.


In [25]:
!pip -q install gdown torch

In [26]:
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


## Download data

In [27]:
import gdown

files = {
    # train
    "wl_2010-2019.npy": "1Ncexf_vB55cpiCeNr-hIRrdpquYaav6B",
    "dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy": "1THbGvO9mVjg_wfZTabRbQBOVpaECl3my",
    "ERA5_adriatic_u10v10sp_2010-2019.npy": "16M1zB54PKkKS6SK8W_U83UURWu1T_AxR",
    "tvec_2010-2019.npy": "161OYs8KQSn3RrXezCNwFvOewXLJe5wBf",
    # test
    "wl_2020.npy": "1iwqd4xzHc98OYqBpGsuUW4SG5AQDtdNR",
    "dist_alt_az_moon-sun_coord13-45_2020_norm.npy": "1cHqyeXtmaiC_3v9uadMD7Y0hONGf2R1q",
    "ERA5_adriatic_u10v10sp_2020.npy": "1AoFAD2viMarikhU5b5Etdsklx08EzKKb",
    "tvec_2020.npy": "1sWoTlJih-mqDdP9TBzhyXTqHHS5Jr9fe",
    # coords
    "lat.npy": "1Mg52QAIo4bfpzJF0dsI8mpZf09tHTzj8",
    "lon.npy": "1wWz0EWbGiBkZ0vfJD8KeZkmVZfmRiYLk",
}

for fname, fid in files.items():
    if not os.path.exists(fname):
        print("Downloading", fname)
        gdown.download(id=fid, output=fname, quiet=False)
    else:
        print("Exists:", fname)

Exists: wl_2010-2019.npy
Exists: dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy
Exists: ERA5_adriatic_u10v10sp_2010-2019.npy
Exists: tvec_2010-2019.npy
Exists: wl_2020.npy
Exists: dist_alt_az_moon-sun_coord13-45_2020_norm.npy
Exists: ERA5_adriatic_u10v10sp_2020.npy
Exists: tvec_2020.npy
Exists: lat.npy
Exists: lon.npy


## Load data (memory-mapped)

In [28]:
lat = np.load("lat.npy")
lon = np.load("lon.npy")

train_wl = np.load("wl_2010-2019.npy", mmap_mode="r")  # (T, 5000)
train_ephem = np.load("dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy", mmap_mode="r")  # (6, T)
train_era5 = np.load("ERA5_adriatic_u10v10sp_2010-2019.npy", mmap_mode="r")  # (3, T, 5, 9)
train_tvec = np.load("tvec_2010-2019.npy", mmap_mode="r")  # (6, T)

test_wl = np.load("wl_2020.npy", mmap_mode="r")
test_ephem = np.load("dist_alt_az_moon-sun_coord13-45_2020_norm.npy", mmap_mode="r")
test_era5 = np.load("ERA5_adriatic_u10v10sp_2020.npy", mmap_mode="r")
test_tvec = np.load("tvec_2020.npy", mmap_mode="r")

T_train = train_wl.shape[0]
N_nodes = train_wl.shape[1]
print("Train shapes:", train_wl.shape, train_ephem.shape, train_era5.shape, train_tvec.shape)
print("Test shapes:", test_wl.shape, test_ephem.shape, test_era5.shape, test_tvec.shape)

Train shapes: (87648, 5000) (6, 87648) (3, 87648, 5, 9) (6, 87648)
Test shapes: (8784, 5000) (6, 8784) (3, 8784, 5, 9) (6, 8784)


## Utilities

In [29]:
def get_era5_coord(lat, lon):
    """
    Match node coordinates to closest ERA5 grid index (safe bounds).
    """
    era5_row, era5_col = 5, 9
    lat_min, lat_max = 44.94972, 45.8
    lon_min, lon_max = 12.12863, 13.81283

    delta_lat = lat_max - lat_min
    delta_lon = lon_max - lon_min

    lon_coord = np.ceil((lon - lon_min) / delta_lon * (era5_col - 1))
    lat_coord = 4 - np.ceil((lat - lat_min) / delta_lat * (era5_row - 1))

    # clip to valid range
    lon_coord = np.clip(lon_coord, 0, era5_col - 1)
    lat_coord = np.clip(lat_coord, 0, era5_row - 1)

    return int(lat_coord), int(lon_coord)

def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


## Normalize ERA5 (using training statistics)

In [30]:
era5_mean = train_era5.mean(axis=(1,2,3), keepdims=True)
era5_std  = train_era5.std(axis=(1,2,3), keepdims=True) + 1e-6

def norm_era5(x):
    return (x - era5_mean) / era5_std

train_era5_norm = norm_era5(train_era5)
test_era5_norm  = norm_era5(test_era5)

print("Train ERA5 norm mean:", train_era5_norm.mean(axis=(1,2,3)))
print("Train ERA5 norm std:", train_era5_norm.std(axis=(1,2,3)))

Train ERA5 norm mean: [-1.43255746e-17 -1.91391983e-17  2.22948601e-15]
Train ERA5 norm std: [0.99999958 0.9999996  1.        ]


## Time encoding

In [31]:
def time_encoding(tvec):
    years  = tvec[0].astype(int)
    months = tvec[1].astype(int)
    days   = tvec[2].astype(int)
    hours  = tvec[3].astype(int)

    leap = (years % 4 == 0) & ((years % 100 != 0) | (years % 400 == 0))
    month_days = np.array([31,28,31,30,31,30,31,31,30,31,30,31])
    cum_days = np.concatenate([[0], np.cumsum(month_days)])

    doy = cum_days[months - 1] + days
    doy += leap & (months > 2)

    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    doy_sin = np.sin(2 * np.pi * doy / 365.0)
    doy_cos = np.cos(2 * np.pi * doy / 365.0)

    return np.stack([hour_sin, hour_cos, doy_sin, doy_cos], axis=1)

train_time_feat = time_encoding(train_tvec)
test_time_feat  = time_encoding(test_tvec)
print("Time features:", train_time_feat.shape)

Time features: (87648, 4)


## Precompute ERA5 grid mapping for nodes

In [32]:
node_era5 = np.array([get_era5_coord(lat[i], lon[i]) for i in range(N_nodes)])
print(node_era5.shape, node_era5[:5])

(5000, 2) [[4 8]
 [4 8]
 [4 8]
 [4 8]
 [4 8]]


## Feature builder

In [33]:
def build_features_pairs(t_idx, n_idx, ephem, time_feat, era5_norm):
    """Aligned pairs: t_idx[k], n_idx[k] -> one sample."""
    t_idx = np.asarray(t_idx)
    n_idx = np.asarray(n_idx)

    ep = ephem[:, t_idx].T
    tf = time_feat[t_idx]
    rows, cols = node_era5[n_idx].T
    er = era5_norm[:, t_idx, rows, cols].T
    latlon = np.stack([lat[n_idx], lon[n_idx]], axis=1)

    return np.concatenate([ep, tf, er, latlon], axis=1)

def build_features_grid(t_idx, n_idx, ephem, time_feat, era5_norm):
    """All combinations of t_idx and n_idx."""
    t_idx = np.asarray(t_idx)
    n_idx = np.asarray(n_idx)

    tt = np.repeat(t_idx, len(n_idx))
    nn = np.tile(n_idx, len(t_idx))

    feats = build_features_pairs(tt, nn, ephem, time_feat, era5_norm)
    return feats, tt, nn

## Train/validation split
- Use last 8760 hours (2019) for validation.

In [34]:
val_hours = 8760
train_idx = np.arange(0, T_train - val_hours)
val_idx = np.arange(T_train - val_hours, T_train)
print("Train hours:", len(train_idx), "Val hours:", len(val_idx))

Train hours: 78888 Val hours: 8760


## Model

In [35]:
class MLP(nn.Module):
    def __init__(self, in_dim=15, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = MLP(in_dim=15, hidden=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

## Training loop (random sampling)

In [36]:
def sample_batch(batch_size, time_pool):
    t_idx = np.random.choice(time_pool, size=batch_size, replace=True)
    n_idx = np.random.randint(0, N_nodes, size=batch_size)
    X = build_features_pairs(t_idx, n_idx, train_ephem, train_time_feat, train_era5_norm)
    y = train_wl[t_idx, n_idx]
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def eval_rmse_sample(num_samples=50000):
    model.eval()
    with torch.no_grad():
        t_idx = np.random.choice(val_idx, size=num_samples, replace=True)
        n_idx = np.random.randint(0, N_nodes, size=num_samples)
        X = build_features_pairs(t_idx, n_idx, train_ephem, train_time_feat, train_era5_norm)
        y = train_wl[t_idx, n_idx]
        pred = model(torch.tensor(X, dtype=torch.float32, device=device)).cpu().numpy()
    return RMSE(y, pred)

epochs = 8
steps_per_epoch = 400
batch_size = 2048

for epoch in range(1, epochs + 1):
    model.train()
    pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch}")
    for _ in pbar:
        X, y = sample_batch(batch_size, train_idx)
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=float(loss))

    val_rmse = eval_rmse_sample()
    print(f"Validation RMSE (sampled): {val_rmse:.4f}")

Epoch 1:   0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipython-input-3313090522.py:34: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  pbar.set_postfix(loss=float(loss))
Epoch 1: 100%|██████████| 400/400 [00:05<00:00, 69.83it/s, loss=0.0304]


Validation RMSE (sampled): 0.1793


Epoch 2: 100%|██████████| 400/400 [00:09<00:00, 40.59it/s, loss=0.0321]


Validation RMSE (sampled): 0.1774


Epoch 3: 100%|██████████| 400/400 [00:06<00:00, 59.98it/s, loss=0.0311]


...

Validation RMSE (sampled): 0.1640


Epoch 7: 100%|██████████| 400/400 [00:04<00:00, 82.76it/s, loss=0.024]


Validation RMSE (sampled): 0.1723


Epoch 8: 100%|██████████| 400/400 [00:05<00:00, 67.56it/s, loss=0.0225]


Validation RMSE (sampled): 0.1593


## Full test RMSE (batched)

In [37]:
def compute_test_rmse(time_block=24, node_block=500):
    model.eval()
    sq_sum = 0.0
    count = 0

    with torch.no_grad():
        for t0 in tqdm(range(0, test_wl.shape[0], time_block), desc="Test time blocks"):
            t_idx = np.arange(t0, min(t0 + time_block, test_wl.shape[0]))
            for n0 in range(0, N_nodes, node_block):
                n_idx = np.arange(n0, min(n0 + node_block, N_nodes))

                X, tt, nn = build_features_grid(t_idx, n_idx, test_ephem, test_time_feat, test_era5_norm)
                X = torch.tensor(X, dtype=torch.float32, device=device)
                pred = model(X).cpu().numpy()

                y_true = test_wl[tt, nn]
                sq_sum += np.sum((pred - y_true) ** 2)
                count += len(pred)

    return np.sqrt(sq_sum / count)

test_rmse = compute_test_rmse(time_block=24, node_block=500)
print(f"Final Test RMSE: {test_rmse:.4f}")

Test time blocks: 100%|██████████| 366/366 [01:12<00:00,  5.04it/s]

Final Test RMSE: 0.1614


## Persistence baseline (reference)

In [38]:
baseline = RMSE(test_wl[:-1], test_wl[1:])
print(f"Persistence baseline RMSE: {baseline:.4f}")

Persistence baseline RMSE: 0.0859
